In [ ]:
pythonimport ee
import geemap
import geopandas as gpd
import matplotlib.pyplot as plt

In [1]:
pip install geemap earthengine-api geopandas

  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ------------------------ --------------- 1.6/2.5 MB 8.1 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 7.8 MB/s  0:00:00
   ---------------------------------------- 0.0/14.8 MB ? eta -:--:--
   --- ------------------------------------ 1.3/14.8 MB 7.0 MB/s eta 0:00:02
   ------- -------------------------------- 2.6/14.8 MB 6.6 MB/s eta 0:00:02
   ----------- ---------------------------- 4.2/14.8 MB 6.9 MB/s eta 0:00:02
   --------------- ------------------------ 5.8/14.8 MB 7.0 MB/s eta 0:00:02
   ------------------- -------------------- 7.1/14.8 MB 6.8 MB/s eta 0:00:02
   --------------------- ------------------ 7.9/14.8 MB 6.4 MB/s eta 0:00:02
   ----------------------- ---------------- 8.7/14.8 MB 6.1 MB/s eta 0:00:02
   --------------------------- ------------ 10.2/14.8 MB 6.2 MB/s eta 0:00:01
   ------------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# 1. Autenticación e inicialización
ee.Authenticate()
ee.Initialize(project='tu-project-id')  # reemplaza con tu project ID

# 2. AOI: Colombia
colombia = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017')\
             .filter(ee.Filter.eq('country_na', 'Colombia'))
aoi = colombia.geometry()

# 3. Colección Sentinel-1
collection = ee.ImageCollection('COPERNICUS/S1_GRD')\
               .filterBounds(aoi)\
               .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
               .select('VV')

# 4. Antes y después temporada de lluvias Colombia
before = collection.filterDate('2023-03-01', '2023-03-31').mosaic()
after  = collection.filterDate('2023-04-01', '2023-05-31').mosaic()

# 5. Detección de inundación
diffSmoothed = after.focal_median(100, 'circle', 'meters')\
                    .subtract(before.focal_median(100, 'circle', 'meters'))
diffThresholded = diffSmoothed.lt(-3)

# 6. Remover agua permanente
jrc_data0 = ee.Image("JRC/GSW1_0/Metadata").select('total_obs').lte(0)
waterMask = ee.Image("JRC/GSW1_0/GlobalSurfaceWater")\
              .select('occurrence').unmask(0).max(jrc_data0).lt(30)
floodedPixels = diffThresholded.updateMask(waterMask)

# 7. Municipios de Colombia
municipios = ee.FeatureCollection('FAO/GAUL/2015/level2')\
               .filter(ee.Filter.eq('ADM0_NAME', 'Colombia'))

# 8. Área inundada por municipio
floodArea = floodedPixels.multiply(ee.Image.pixelArea())
stats = floodArea.reduceRegions(
    collection=municipios,
    reducer=ee.Reducer.sum(),
    scale=100
)

# 9. Descargar directo a tu PC (cambia la ruta si quieres)
output_path = r'C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Stress Test\Data\flood_municipios.shp'
geemap.ee_to_shp(stats, filename=output_path)
print('✅ Archivo guardado en:', output_path)

# 10. Análisis en Python
gdf = gpd.read_file(output_path)
gdf['area_total'] = gdf.geometry.to_crs('EPSG:3116').area
gdf['susceptibilidad'] = (gdf['sum'] / gdf['area_total']) * 100

# Ranking top 20
ranking = gdf[['ADM2_NAME', 'ADM1_NAME', 'susceptibilidad']]\
            .sort_values('susceptibilidad', ascending=False)
print(ranking.head(20))

# Mapa
gdf.plot(column='susceptibilidad',
         cmap='YlOrRd',
         legend=True,
         figsize=(10, 14),
         title='Susceptibilidad a inundaciones por municipio - Colombia 2023')
plt.tight_layout()